# Experiment 1: baseline confusion classifier (TF-IDF + Logistic Regression)

*J26-IT-304 PP1. Notebook 01, sections 1-2 (setup, load and inspect).*

## 1. Setup and provenance

**Objective.** Train a TF-IDF + Logistic Regression baseline that classifies a student's answer
text as *confused* / *understood*. This is a supervised, machine-learning-based text
classification task (IRWA Lecture 06). The pipeline is: text preprocessing, then feature
extraction (TF-IDF), then a classifier, evaluated with a validation-set approach.

**Important caveat.** "Confused" is a **proxy** built from the dataset's answer-correctness
label, not a direct measurement of confusion. The full caveat comes in section 5.

**Why this cell.** A result is only reproducible if we know exactly which library versions, seed
and dataset commit produced it. The scikit-learn version matters most, because the tutor service
must unpickle the final model under the same version. The dataset revision is a pinned commit
SHA, so a later run fetches exactly this data and not "latest".

In [1]:
import sys
import numpy as np
import pandas as pd
import sklearn
import huggingface_hub

from confusion_baseline import SEED
from confusion_baseline.data import DATASET_REPO, DATASET_REVISION, load_raw

print("python            :", sys.version.split()[0])
print("pandas            :", pd.__version__)
print("numpy             :", np.__version__)
print("scikit-learn      :", sklearn.__version__)
print("huggingface_hub   :", huggingface_hub.__version__)
print("SEED              :", SEED)
print("DATASET_REPO      :", DATASET_REPO)
print("DATASET_REVISION  :", DATASET_REVISION)

python            : 3.13.16
pandas            : 3.0.6
numpy             : 2.5.4
scikit-learn      : 1.9.1
huggingface_hub   : 2.2.0
SEED              : 42
DATASET_REPO      : nitinnovtech-ai-space/JavaQuestionSynthFinal
DATASET_REVISION  : e29cfdd4ec4179f54b2ea0409329207104de8bed


## 2. Load and inspect

**Why this cell.** `load_raw()` is the single entry point to the data, shared with the training
script so notebook and script cannot drift. It assigns `row_id` (position in the raw file, the
stable key for all later units) before anything else, and it moves every `FollowUp`-derived
column into a separate `followup_quarantine` frame. The tutor-feedback text in `FollowUp`
reveals the label, so the working frame must never contain it. The quarantine lets a later
unit show one example of the leak without the leak ever being in `df`.

In [2]:
df, followup_quarantine, load_report = load_raw()

print("raw file format :", load_report["format"], load_report["files"])
print("shape           :", df.shape)
print("columns         :", list(df.columns))
print()
print("dtypes:")
print(df.dtypes.to_string())

raw file format : parquet ['data/train-00000-of-00001.parquet']
shape           : (1078, 4)
columns         : ['row_id', 'Question', 'Answer', 'Correctness']

dtypes:
row_id         int64
Question         str
Answer           str
Correctness      str


**Why this cell.** First look at the records, and a count of missing values per column. Empty
or null answers would need a decision in the cleaning unit, so we want to know now whether any exist.

In [3]:
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)
display(df.head(5))
print("missing values per column:")
print(df.isna().sum().to_string())

,row_id,Question,Answer,Correctness
0,0,What is the difference between an abstract class and an interface in Java?,"An abstract class can have both abstract methods and concrete methods, where...",Correct
1,1,What is the difference between an abstract class and an interface in Java?,Abstract classes are used to provide common functionality for all subclasses...,Partial Correct
2,2,What is the difference between an abstract class and an interface in Java?,An abstract class is used for defining blueprints and can contain both imple...,Wrong
3,3,Explain the concept of exception handling in Java.,Exception handling in Java is done using try-catch blocks. If an exception i...,Correct
4,4,Explain the concept of exception handling in Java.,"Java exceptions are for catching errors like in runtime, and it uses catch b...",Partial Correct


missing values per column:
row_id         0
Question       0
Answer         0
Correctness    0


**Why this cell.** Exact duplicates can leak between train and validation splits and inflate the
score. Two counts are printed on purpose, because they answer different questions: the working
frame has no `FollowUp` columns, so rows that differ only in their follow-up text look identical
there. The count on the full raw record (computed inside `load_raw()`, where `FollowUp` is still
present) tells us how many of those are true duplicates.

In [4]:
dup_working = int(df.drop(columns="row_id").duplicated().sum())
dup_full = load_report["duplicates_full_record_incl_followup"]
print("exact duplicates, WORKING frame (all columns except row_id, no FollowUp):", dup_working)
print("exact duplicates, FULL raw record (INCLUDING FollowUp)                  :", dup_full)

exact duplicates, WORKING frame (all columns except row_id, no FollowUp): 2
exact duplicates, FULL raw record (INCLUDING FollowUp)                  : 2


**Why this cell.** A schema check on the label column only. `repr()` makes stray whitespace or
case variants visible, which would silently create extra classes. This prints raw strings only;
the distribution analysis is the next unit.

In [5]:
for value, n in df["Correctness"].value_counts(dropna=False).items():
    print(f"{value!r:20} {n}")

'Correct'            422
'Wrong'              367
'Partial Correct'    289


**Why this cell.** The spec's description of the dataset is second-hand. This table puts what the
spec says next to what was actually found, so any correction is explicit and evidenced.

In [6]:
spec_labels = {"Correct", "Partial Correct", "Wrong"}
found_labels = set(df["Correctness"].dropna().unique())
fu_cols = load_report["followup_columns_raw"]
fu_struct = load_report["followup_structure"]
fu_keys = [c for c in followup_quarantine.columns if c != "row_id"]

def row(item, spec, found, ok):
    return {"item": item, "spec says": spec, "found": found, "match": "yes" if ok else "no"}

schema = pd.DataFrame([
    row("row count", "~1,078", len(df), 1000 <= len(df) <= 1150),
    row("Question", "column present", f"present, dtype {df['Question'].dtype}" if "Question" in df else "absent", "Question" in df),
    row("Answer", "column present", f"present, dtype {df['Answer'].dtype}" if "Answer" in df else "absent", "Answer" in df),
    row("Correctness", "column present", f"present, dtype {df['Correctness'].dtype}" if "Correctness" in df else "absent", "Correctness" in df),
    row("Correctness values", sorted(spec_labels), sorted(found_labels), found_labels == spec_labels),
    row("FollowUp Part1", "field present", f"{fu_cols} ({fu_struct}) -> {'followup_Part1' in fu_keys}", "followup_Part1" in fu_keys),
    row("FollowUp Part2", "field present", f"{fu_cols} ({fu_struct}) -> {'followup_Part2' in fu_keys}", "followup_Part2" in fu_keys),
])
display(schema)

,item,spec says,found,match
0,row count,"~1,078",1078,yes
1,Question,column present,"present, dtype str",yes
2,Answer,column present,"present, dtype str",yes
3,Correctness,column present,"present, dtype str",yes
4,Correctness values,"[Correct, Partial Correct, Wrong]","[Correct, Partial Correct, Wrong]",yes
5,FollowUp Part1,field present,['FollowUp Question'] ({'FollowUp Question': 'dict'}) -> True,yes
6,FollowUp Part2,field present,['FollowUp Question'] ({'FollowUp Question': 'dict'}) -> True,yes


**Why this cell.** Shape and column names of the quarantine only. Its contents are deliberately
not printed here; the leakage demonstration belongs to the next unit.

In [7]:
print("followup_quarantine shape  :", followup_quarantine.shape)
print("followup_quarantine columns:", list(followup_quarantine.columns))

followup_quarantine shape  : (1078, 3)
followup_quarantine columns: ['row_id', 'followup_Part1', 'followup_Part2']


**Why this cell.** A hard guarantee instead of a promise: if any `FollowUp`-derived column is in
the working frame, the notebook stops here.

In [8]:
leaked = [c for c in df.columns if 'follow' in c.lower()]
assert not leaked, leaked
print("PASS: no FollowUp-derived column in the working frame:", list(df.columns))

PASS: no FollowUp-derived column in the working frame: ['row_id', 'Question', 'Answer', 'Correctness']
